# N2 — The Kalman Filter: Recursive State Estimation from Scratch

This notebook implements the Kalman filter from first principles — no estimation libraries, just NumPy matrices. We build intuition for the predict/update loop, watch the Kalman gain converge, tune noise parameters, and see the filter coast gracefully through sensor dropout.

All plotting and animation code lives in [`n2_viz.py`](n2_viz.py) — the cells here contain only the filter math.

**What you will learn:**
1. The predict/update loop: predict widens belief, update narrows it
2. The Kalman gain as a "trust knob" between prediction and measurement
3. How to tune process noise **Q** and measurement noise **R**
4. Matrix-form predict/update for arbitrary linear systems
5. Coasting through sensor dropout with honest uncertainty growth

```
        PREDICT                          UPDATE
   (motion model)                   (measurement)

  x̂⁻ = F · x̂⁺                    K = P⁻ · Hᵀ · (H · P⁻ · Hᵀ + R)⁻¹
  P⁻ = F · P⁺ · Fᵀ + Q            x̂⁺ = x̂⁻ + K · (z - H · x̂⁻)
                                    P⁺ = (I - K · H) · P⁻
       │                                  │
       ▼                                  ▼
  Belief WIDENS                    Belief NARROWS
  (less certain)                   (more certain)
```

## 1. Setup

In [ ]:
%pip install numpy matplotlib ipywidgets --quiet

In [ ]:
import numpy as np

import n2_viz   # all plotting/animation helpers live here — see n2_viz.py

np.random.seed(42)

### How the plotting works

Every figure in this notebook is drawn with **matplotlib** by a function in [`n2_viz.py`](n2_viz.py) — the cells here just compute filter results and pass them in. Three patterns cover everything:

- **Static plots** (`plot_gain_convergence`, `plot_qr_comparison`) — create a figure and axes with `plt.subplots()`, draw lines/scatter/shaded ±2σ bands onto the axes, then `plt.show()` renders it inline.
- **Animations** (`animate_1d`, `animate_2d`, `animate_dropout`) — set up the figure once, then `matplotlib.animation.FuncAnimation` calls an `update(frame)` function that moves the artists (lines, points, covariance `Ellipse` patches) one timestep at a time. The animation is converted to an interactive HTML/JavaScript player with `anim.to_jshtml()`, so you can play, pause, and scrub without any video backend.
- **Interactive explorer** (`interactive_qr_explorer`) — `ipywidgets` sliders re-run the filter and redraw the matplotlib figure every time you drag, so the plot always reflects the current Q and R.

Uncertainty is visualized the same way throughout: the 2×2 position covariance is turned into a **2σ confidence ellipse** (eigenvectors give the orientation, eigenvalues the axis lengths) drawn with `matplotlib.patches.Ellipse`.

## 2. The Predict–Update Loop

Every Kalman filter runs the same two-step loop — **predict**, then **update** — maintaining a probability distribution (the *belief*) over the system's hidden state:

**Predict** — use a motion model to propagate the belief forward in time. Because the model is imperfect, uncertainty *grows*.

**Update** — incorporate a new measurement to correct the prediction. New information *shrinks* uncertainty.

When the motion model and measurement model are both **linear** and the noise is **Gaussian**, this loop has an exact closed-form solution — the **Kalman filter**. The belief stays Gaussian forever, fully described by a mean vector $\hat{x}$ and covariance matrix $P$.

| Step | Equation | Meaning |
|------|----------|---------|
| **Predict state** | $\hat{x}^- = F \hat{x}^+$ | Propagate the mean through the motion model |
| **Predict covariance** | $P^- = F P^+ F^\top + Q$ | Propagate uncertainty; $Q$ adds process noise |
| **Kalman gain** | $K = P^- H^\top (H P^- H^\top + R)^{-1}$ | Balance trust between prediction and measurement |
| **Update state** | $\hat{x}^+ = \hat{x}^- + K(z - H\hat{x}^-)$ | Correct the prediction toward the measurement |
| **Update covariance** | $P^+ = (I - KH) P^-$ | Shrink uncertainty by the information gained |

where:
- $F$ — state transition matrix (motion model)
- $H$ — measurement matrix (what the sensor observes)
- $Q$ — process noise covariance (how much we distrust the model)
- $R$ — measurement noise covariance (how much we distrust the sensor)
- $z$ — the actual measurement

## 3. 1D Kalman Filter — Tracking a Moving Object

The simplest possible case: an object moving at roughly constant velocity, measured by a noisy position-only sensor. The state is $[\,x,\; \dot{x}\,]^\top$ — position **and** velocity — even though we only ever measure position:

$$F = \begin{bmatrix} 1 & \Delta t \\ 0 & 1 \end{bmatrix}, \qquad H = \begin{bmatrix} 1 & 0 \end{bmatrix}$$

The filter below is the entire algorithm — five lines of math inside the loop. Everything else is bookkeeping so we can plot how the belief evolves.

In [ ]:
def kalman_filter(measurements, F, H, Q, R, x0, P0):
    """Run a linear Kalman filter over a sequence of measurements.

    Returns a history dict so we can inspect how the estimate,
    uncertainty, and Kalman gain evolve over time.
    """
    n, dim_x = len(measurements), F.shape[0]
    hist = {
        "x":      np.zeros((n, dim_x)),              # posterior state estimate
        "P":      np.zeros((n, dim_x, dim_x)),       # posterior covariance
        "K":      np.zeros((n, dim_x, H.shape[0])),  # Kalman gain
        "P_pred": np.zeros((n, dim_x, dim_x)),       # covariance BEFORE the update
    }

    x, P = x0.copy(), P0.copy()

    for k in range(n):
        # --- PREDICT: push the belief through the motion model ---
        x = F @ x                # where do we THINK the object is now?
        P = F @ P @ F.T + Q      # uncertainty GROWS — the model is imperfect
        hist["P_pred"][k] = P

        # --- UPDATE: correct the prediction with the new measurement ---
        y = measurements[k] - H @ x        # innovation: how wrong was the prediction?
        S = H @ P @ H.T + R                # how wrong did we EXPECT to be?
        K = P @ H.T @ np.linalg.inv(S)     # gain: trust knob between model and sensor
        x = x + K @ y                      # nudge the estimate toward the measurement
        P = (np.eye(dim_x) - K @ H) @ P    # uncertainty SHRINKS — we gained information

        hist["x"][k], hist["P"][k], hist["K"][k] = x, P, K

    return hist

In [ ]:
# ── Tunable parameters — change these and re-run! ───────────────────────────
process_noise_std     = 0.5    # Q scale: low = trust the model (smooth), high = trust the sensor (responsive)
measurement_noise_std = 10.0   # R scale: how noisy the sensor is
# ─────────────────────────────────────────────────────────────────────────────

# Ground truth: constant-velocity motion. The sensor reports position + Gaussian noise.
dt, num_steps, true_velocity = 1.0, 80, 2.0
true_positions = true_velocity * dt * np.arange(num_steps)
measurements = true_positions + np.random.normal(0, measurement_noise_std, num_steps)

F = np.array([[1, dt],
              [0,  1]])                 # constant-velocity motion model
H = np.array([[1.0, 0.0]])              # the sensor sees position only

# Q: discrete white-noise acceleration model — allows small unmodeled accelerations
Q = np.array([[dt**4/4, dt**3/2],
              [dt**3/2, dt**2  ]]) * process_noise_std**2
R = np.array([[measurement_noise_std**2]])

x0 = np.array([0.0, 0.0])               # initial guess: at the origin, not moving...
P0 = np.diag([500.0, 50.0])             # ...but we are VERY unsure about that

hist = kalman_filter(measurements.reshape(-1, 1), F, H, Q, R, x0, P0)

print(f"True velocity:            {true_velocity:.1f} m/s")
print(f"Final estimated velocity: {hist['x'][-1, 1]:.2f} m/s  (never measured directly!)")
print(f"Final position error:     {abs(hist['x'][-1, 0] - true_positions[-1]):.2f} m")

In [ ]:
n2_viz.animate_1d(true_positions, measurements, hist, true_velocity)

## 4. Kalman Gain Convergence

The Kalman gain $K$ starts large (the filter trusts measurements heavily because its initial state estimate is uncertain) and converges to a steady-state value as the filter becomes more confident. This convergence depends only on $F$, $H$, $Q$, and $R$ — not the actual measurements.

- **High gain** → filter tracks measurements closely (responsive but noisy)
- **Low gain** → filter trusts its model more (smooth but slow to react)

The right panel shows the heartbeat of the filter: predicted $\sigma$ (after predict, wider) vs posterior $\sigma$ (after update, narrower), step after step.

In [ ]:
n2_viz.plot_gain_convergence(hist)

## 5. Q vs R Tuning: Stiff, Balanced, and Floppy

The ratio $Q / R$ controls the filter's personality:

| Setting | Q/R Ratio | Behavior | Analogy |
|---------|-----------|----------|---------|
| **Stiff** | Low Q, high R | Trusts the model; smooths aggressively | "I know the physics, the sensor is noisy" |
| **Balanced** | Moderate | Tracks the signal without excess noise | The sweet spot |
| **Floppy** | High Q, low R | Trusts measurements; tracks every wiggle | "The sensor is gospel, the model is approximate" |

To see this, we run three filters on the **same** noisy data — a sinusoidal trajectory where the constant-velocity model is a poor fit, so the choice of $Q$ really matters.

In [ ]:
# ── Tunable parameters ───────────────────────────────────────────────────────
meas_noise_sin = 8.0                                       # shared sensor noise
q_configs = [("Stiff", 0.01), ("Balanced", 0.5), ("Floppy", 5.0)]   # Q scales
# ─────────────────────────────────────────────────────────────────────────────

# Sinusoid + drift: the constant-velocity model is WRONG here, so Q matters
np.random.seed(42)
time_sin = np.arange(120)
true_pos_sin = 0.5 * time_sin + 15 * np.sin(0.08 * time_sin)
meas_sin = true_pos_sin + np.random.normal(0, meas_noise_sin, len(time_sin))

R_sin = np.array([[meas_noise_sin**2]])

# Same data, three different Q values — only the filter's "personality" changes
results = []
for label, q_scale in q_configs:
    Q_sin = np.array([[dt**4/4, dt**3/2],
                      [dt**3/2, dt**2  ]]) * q_scale**2
    h = kalman_filter(meas_sin.reshape(-1, 1), F, H, Q_sin, R_sin,
                      np.array([0.0, 0.0]), np.diag([500.0, 50.0]))
    rmse = np.sqrt(np.mean((h["x"][:, 0] - true_pos_sin)**2))
    results.append((f"{label} (Q scale = {q_scale})", h["x"],
                    np.sqrt(h["P"][:, 0, 0]), rmse))
    print(f"{label:10s} Q scale = {q_scale:<5}  →  RMSE = {rmse:.2f} m")

In [ ]:
n2_viz.plot_qr_comparison(true_pos_sin, meas_sin, results)

### Interactive Explorer: Drag Q and R

Now it's your turn — sweep Q and R continuously with the sliders. **The measurements stay fixed**; the sliders only change the filter's internal Q and R, so you're exploring what happens when the filter's assumptions match or mis-match reality:

- **Crank Q up** — the filter admits the model is poor, trusts measurements more, Kalman gain rises
- **Crank R up past the true noise (8)** — the filter thinks the sensor is worse than it is, over-smooths, lags behind curves
- **Crank R down below the true noise** — the filter over-trusts noisy measurements, tracks every wiggle
- **Set R ≈ 8 (true noise)** — the filter's belief matches reality, RMSE is minimized

In [ ]:
n2_viz.interactive_qr_explorer(kalman_filter, F, H, true_pos_sin, meas_sin,
                               dt=dt, true_noise_std=meas_noise_sin)

## 6. 2D Constant-Velocity Tracking (Full Matrix Form)

Now we generalize to 2D: tracking an object moving on a plane. The state is $[\,x,\, y,\, \dot{x},\, \dot{y}\,]^\top$ and the sensor observes position only:

$$F = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}, \qquad H = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}$$

Note that the `predict` and `update` methods below are **identical math** to the 1D case — only the matrix sizes changed. This is the same filter structure that appears in N4 (Multi-Object Tracker), where each YOLO detection's bounding-box center is tracked independently.

In [ ]:
class KalmanFilter2D:
    """General-purpose linear Kalman filter in matrix form — just NumPy.

    Split into separate predict() / update() methods because real systems
    don't always have a measurement for every prediction (see Section 7).
    """
    def __init__(self, F, H, Q, R, x0, P0):
        self.F, self.H, self.Q, self.R = F, H, Q, R
        self.x = x0.copy()
        self.P = P0.copy()

    def predict(self):
        # Same two lines as the 1D filter — belief moves forward, uncertainty grows
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def update(self, z):
        # Same five lines as the 1D filter — correct toward z, uncertainty shrinks
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(len(self.x)) - K @ self.H) @ self.P

In [ ]:
# ── Tunable parameters ───────────────────────────────────────────────────────
q_2d          = 0.3    # process noise: how much unmodeled acceleration to expect
meas_noise_2d = 5.0    # sensor noise on the position measurements
# ─────────────────────────────────────────────────────────────────────────────

# Ground truth: a figure-eight — constantly turning, so the CV model is always
# slightly wrong and the filter must keep correcting
np.random.seed(42)
num_steps_2d, dt_2d = 200, 0.5
t_2d = np.arange(num_steps_2d) * dt_2d
true_x = 40 * np.sin(0.04 * t_2d)
true_y = 20 * np.sin(0.08 * t_2d)
meas_x = true_x + np.random.normal(0, meas_noise_2d, num_steps_2d)
meas_y = true_y + np.random.normal(0, meas_noise_2d, num_steps_2d)

F_2d = np.array([[1, 0, dt_2d, 0    ],
                 [0, 1, 0,     dt_2d],
                 [0, 0, 1,     0    ],
                 [0, 0, 0,     1    ]])
H_2d = np.array([[1, 0, 0, 0],
                 [0, 1, 0, 0]], dtype=float)

# Same white-noise-acceleration Q as 1D, block-structured for [x, y, vx, vy]
Q_2d = q_2d**2 * np.array([
    [dt_2d**4/4, 0,          dt_2d**3/2, 0         ],
    [0,          dt_2d**4/4, 0,          dt_2d**3/2],
    [dt_2d**3/2, 0,          dt_2d**2,   0         ],
    [0,          dt_2d**3/2, 0,          dt_2d**2  ]])
R_2d = meas_noise_2d**2 * np.eye(2)

kf = KalmanFilter2D(F_2d, H_2d, Q_2d, R_2d,
                    x0=np.array([meas_x[0], meas_y[0], 0.0, 0.0]),  # start at first measurement
                    P0=np.diag([100.0, 100.0, 25.0, 25.0]))

# The classic loop: predict, then update with each measurement as it arrives
est_2d = np.zeros((num_steps_2d, 4))
cov_2d = np.zeros((num_steps_2d, 4, 4))
for k in range(num_steps_2d):
    kf.predict()
    kf.update(np.array([meas_x[k], meas_y[k]]))
    est_2d[k], cov_2d[k] = kf.x, kf.P

rmse_2d = np.sqrt(np.mean((est_2d[:, 0] - true_x)**2 + (est_2d[:, 1] - true_y)**2))
print(f"2D tracking RMSE:  {rmse_2d:.2f} m")
print(f"Noise reduction:   {meas_noise_2d / rmse_2d:.1f}× improvement over raw measurements")

In [ ]:
n2_viz.animate_2d(t_2d, true_x, true_y, meas_x, meas_y, est_2d, cov_2d)

## 7. Sensor Dropout: Coasting Through Measurement Loss

What happens when the sensor goes dark? In the real world, measurements are lost all the time — occlusion, sensor failure, dropped packets. A well-tuned Kalman filter handles this gracefully by **coasting** on its motion model alone:

- The **predict** step still runs (the filter's best guess propagates forward)
- The **update** step is skipped (no measurement available)
- **Uncertainty grows** monotonically — the filter knows it's becoming less certain
- When measurements resume, the filter **snaps back** — a large Kalman gain corrects the accumulated drift

This is exactly how the multi-object tracker in N4 keeps track of vehicles that are briefly occluded. Note the only change from the loop above: `update()` is inside an `if`.

In [ ]:
# ── Tunable parameters ───────────────────────────────────────────────────────
dropout_start, dropout_end = 70, 130   # sensor goes dark for steps 70–129
# ─────────────────────────────────────────────────────────────────────────────

kf = KalmanFilter2D(F_2d, H_2d, Q_2d, R_2d,
                    x0=np.array([meas_x[0], meas_y[0], 0.0, 0.0]),
                    P0=np.diag([100.0, 100.0, 25.0, 25.0]))

est_drop = np.zeros((num_steps_2d, 4))
cov_drop = np.zeros((num_steps_2d, 4, 4))
for k in range(num_steps_2d):
    kf.predict()                                  # ALWAYS predict...
    if not (dropout_start <= k < dropout_end):
        kf.update(np.array([meas_x[k], meas_y[k]]))   # ...but only update when we have a measurement
    est_drop[k], cov_drop[k] = kf.x, kf.P

err_drop = np.sqrt((est_drop[:, 0] - true_x)**2 + (est_drop[:, 1] - true_y)**2)
sigma_drop = np.sqrt(cov_drop[:, 0, 0] + cov_drop[:, 1, 1])

print(f"Dropout window: steps {dropout_start}–{dropout_end} "
      f"({(dropout_end - dropout_start) * dt_2d:.0f} s blackout)")
print(f"Peak error during dropout:    {err_drop[dropout_start:dropout_end].max():.1f} m")
print(f"Error 5 steps after recovery: {err_drop[min(dropout_end + 5, num_steps_2d - 1)]:.1f} m")
print(f"Peak uncertainty (σ):         {sigma_drop[dropout_start:dropout_end].max():.1f} m")

In [ ]:
n2_viz.animate_dropout(t_2d, true_x, true_y, meas_x, meas_y, est_drop, cov_drop,
                       dropout_start, dropout_end)

## Summary

We built the Kalman filter from scratch — no estimation libraries, just matrices:

1. **The predict/update loop** — motion model widens belief, measurement narrows it
2. **1D tracking** — estimated position *and velocity* from noisy position-only measurements
3. **Kalman gain convergence** — the gain starts large and settles to steady state as confidence stabilizes
4. **Q/R tuning** — stiff (smooth, slow), balanced, and floppy (noisy, responsive) personalities
5. **2D matrix form** — the identical math generalizes to a 4-state $[x, y, \dot{x}, \dot{y}]$ tracker
6. **Sensor dropout** — the filter coasts on its model, uncertainty grows honestly, and the gain spikes on recovery to snap the estimate back

### Key Takeaways

- The Kalman filter is **optimal** for linear-Gaussian systems — but only as good as its noise parameters ($Q$ and $R$) match reality
- **Predict widens, update narrows** — this rhythm is the heartbeat of all Bayesian estimation
- The Kalman gain is a **trust knob**: $K \approx 1$ means "follow the measurement", $K \approx 0$ means "trust the model"
- **Coasting** through measurement loss is not a failure mode — it's a designed feature

### What's Next

- **N3** — LiDAR 3D Tracking: this same constant-velocity Kalman filter tracks real objects detected in LiDAR point clouds
- **N4** — Multi-Object Tracker: one Kalman filter per tracked object, with the Hungarian algorithm for data association — turning per-frame YOLO detections into persistent tracks

### References
- Kalman, R. E. (1960) — *A New Approach to Linear Filtering and Prediction Problems*
- Thrun, Burgard, & Fox — *Probabilistic Robotics*, Chapters 2–3
- Welch & Bishop (2006) — *An Introduction to the Kalman Filter* (UNC TR 95-041) — the classic accessible tutorial